# Tugas Mandiri Pertemuan 11 - Machine Learning dengan Spark MLlib

**Nama:** Muhamad Dimas Adji Saputra
**NPM:** 2505060078
**Mata Kuliah:** Praktikum Big Data

---

In [2]:
from pyspark.sql import SparkSession

# Membuat SparkSession
spark = SparkSession.builder.appName("TugasMandiri11").master("local[*]").getOrCreate()

# Membaca dataset (pastikan file tugas11_penjualan.csv ada di folder yang sama)
df_tugas = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df_tugas.show(5)
print("Total baris:", df_tugas.count())

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows

Total baris: 2500


### A. Feature Engineering
Gunakan `StringIndexer` untuk mengubah kolom `nama_menu`, `cabang`, dan `waktu_pesan` menjadi indeks numerik. Gabungkan seluruh fitur (`qty`, `harga_satuan`, dan ketiga kolom hasil indexing) menjadi satu kolom `features` menggunakan `VectorAssembler`.

In [3]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# 1. Fit & Transform StringIndexer (simpan objek indexer untuk dipakai lagi di Poin E)
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx").fit(df_tugas)
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df_tugas)
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx").fit(df_tugas)

df_idx = indexer_menu.transform(df_tugas)
df_idx = indexer_cabang.transform(df_idx)
df_idx = indexer_waktu.transform(df_idx)

# 2. VectorAssembler untuk menggabungkan 5 fitur
assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "menu_idx", "cabang_idx", "waktu_idx"],
    outputCol="features"
)
df_features = assembler.transform(df_idx)
df_features.select("features", "total_penjualan").show(5, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



### B. Train/Test Split & Pelatihan Model
Bagi data menjadi 75% training dan 25% testing (`seed=101`). Latih model `LinearRegression` dengan `labelCol="total_penjualan"`.

In [4]:
from pyspark.ml.regression import LinearRegression

# Split data 75% dan 25%
train_data, test_data = df_features.randomSplit([0.75, 0.25], seed=101)

# Melatih model Regresi Linear
lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_utama = lr.fit(train_data)

print("Model utama berhasil dilatih!")

26/11/18 19:31:58 WARN Instrumentation: [5af63b34] regParam is zero, which might cause numerical instability and overfitting.
26/11/18 19:31:58 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/11/18 19:31:58 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Model utama berhasil dilatih!


### C. Evaluasi Model
Hitung dan tampilkan **RMSE** dan **R²** dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?

In [5]:
from pyspark.ml.evaluation import RegressionEvaluator

# Membuat prediksi pada data testing
predictions = model_utama.transform(test_data)

# Evaluasi RMSE dan R2
evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}")

RMSE : Rp 10,901
R²   : 0.9107


**Interpretasi Model:**
Model ini sangat baik untuk dipakai memprediksi omzet karena memiliki nilai R² yang sangat tinggi (mendekati 1.0). Artinya, variasi data penjualan hampir sepenuhnya berhasil dipelajari dan ditebak dengan akurat oleh model. Kesalahan rata-rata (RMSE) juga tergolong kecil dibandingkan nominal total omzet per transaksi.

### D. Eksperimen Perbandingan Fitur
Latih **model kedua** yang HANYA menggunakan fitur `qty` dan `harga_satuan` (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?

In [6]:
# Membuat assembler baru khusus 2 fitur
assembler_sedikit = VectorAssembler(inputCols=["qty", "harga_satuan"], outputCol="features_sedikit")
df_features_sedikit = assembler_sedikit.transform(df_idx)

# Split ulang menggunakan seed yang sama persis
train_sedikit, test_sedikit = df_features_sedikit.randomSplit([0.75, 0.25], seed=101)

# Melatih model kedua
model_kedua = LinearRegression(featuresCol="features_sedikit", labelCol="total_penjualan").fit(train_sedikit)

# Evaluasi model kedua
pred_sedikit = model_kedua.transform(test_sedikit)
r2_kedua = evaluator_r2.evaluate(pred_sedikit)

print(f"R² Model 5 Fitur : {r2:.4f}")
print(f"R² Model 2 Fitur : {r2_kedua:.4f}")

26/11/18 19:35:28 WARN Instrumentation: [07862393] regParam is zero, which might cause numerical instability and overfitting.


R² Model 5 Fitur : 0.9107
R² Model 2 Fitur : 0.9108


**Kesimpulan Eksperimen:**
Menambahkan informasi `nama_menu`, `cabang`, dan `waktu_pesan` ternyata **tidak** meningkatkan akurasi prediksi secara signifikan (nilai R² model kedua hampir sama identik dengan model utama). Hal ini sangat logis karena secara matematis, `total_penjualan` murni terbentuk dari perkalian antara *quantity* (qty) dan *harga satuan*, sedangkan cabang dan waktu pemesanan tidak berpengaruh langsung terhadap hitungan omzet per pesanan tersebut pada dataset buatan ini.

### E. Prediksi Skenario Baru
Buat sebuah DataFrame baru berisi **3 skenario pesanan (data dummy)** (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan `StringIndexer` dan `VectorAssembler` yang sama seperti data training — perhatikan: gunakan model `StringIndexer` yang **sama** yang sudah di-`fit()` sebelumnya, jangan membuat `StringIndexer` baru). Tampilkan hasil prediksi `total_penjualan` untuk ketiga skenario tersebut.

In [7]:
# 1. Membuat DataFrame skenario baru
data_skenario = [
    ("Kopi Susu", "Magelang", "Pagi", 4, 15000),
    ("Roti Bakar", "Yogyakarta", "Malam", 2, 22000),
    ("Es Teh", "Semarang", "Siang", 10, 10000)
]
df_skenario = spark.createDataFrame(data_skenario, ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"])

# 2. Tahap ETL: Menggunakan transform() dari objek indexer yang SUDAH di-fit di Poin A!
df_skenario_idx = indexer_menu.transform(df_skenario)
df_skenario_idx = indexer_cabang.transform(df_skenario_idx)
df_skenario_idx = indexer_waktu.transform(df_skenario_idx)

# Menggunakan assembler yang sama dari Poin A
df_skenario_features = assembler.transform(df_skenario_idx)

# 3. Prediksi menggunakan model utama (5 fitur)
prediksi_skenario = model_utama.transform(df_skenario_features)
prediksi_skenario.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show()

+----------+----------+-----------+---+------------+-----------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|       prediction|
+----------+----------+-----------+---+------------+-----------------+
| Kopi Susu|  Magelang|       Pagi|  4|       15000|66504.07418566766|
|Roti Bakar|Yogyakarta|      Malam|  2|       22000|43820.47427272107|
|    Es Teh|  Semarang|      Siang| 10|       10000|184660.8675989256|
+----------+----------+-----------+---+------------+-----------------+



---
### Menutup SparkSession

In [8]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
